# Confusion SAST: Interactive Explorer

This notebook walks through the core workflow: discovering targets,
extracting analysis graphs, scanning for vulnerabilities, and exploring
results interactively.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from confusion_sast.notebook import *

In [2]:
t = targets()
target = t.r01.e03

## Target Selection

The `targets()` registry auto-discovers exercises from the `webapp/` directory.
Each section contains progressive exercises, from baseline to fixed.

In [3]:
t

TargetRegistry(/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp)
  r01: r01_input_source_confusion
    e00: e00_baseline
    e01: e01_dual_parameter_easier
    e02: e02_delivery_fee
    e03: e03_order_overwrite
    e04: e04_negative_tip
    e05: e05_unlimited_refund
    e06: e06_signup_token_swap
    e07: e07_signup_bonus
    e08: e08_fixed_final_version
  r02: r02_authentication_confusion
    e01: e01_session_hijack
    e02: e02_credit_top_ups
    e03: e03_fake_header_refund
    e04: e04_manager_mode
    e05: e05_session_overwrite
    e06: e06_fixed_final_version
  r03: r03_authorization_confusion
    e01: e01_dual_auth_refund
    e02: e02_cart_swap_checkout
    e03: e03_menu_edits
    e04: e04_body_override_orders
    e05: e05_failed_update_leaks
    e06: e06_domain_token_any_mailbox
    e07: e07_token_swap_hijack
    e08: e08_fixed_final_version
  r04: r04_cardinality_confusion
    e01: e01_coupon_stacking
    e02: e02_zero_quantity
    e03: e03_duplicate_coupons
    e04: e04_batch_refund_bypass
    e05: e05_two_ids_bypass
  r05: r05_normalization_issues

In [4]:
# Section and exercise objects are richer than raw Paths.
print(t.r01)
print(t.r01.e01)
print(type(t.r01.e01).__name__, "->", Path(t.r01.e01))
print(type(t.r01.e01.path).__name__, "->", t.r01.e01.path)
print("batch_load(t.r01) loads one section; batch_load(t) loads everything")

Section(r01: 9 exercises [e00, e01, e02, e03, e04, e05, e06, e07, e08])
Exercise(r01/e01: e01_dual_parameter_easier)
Exercise -> /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e01_dual_parameter_easier
PosixPath -> /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e01_dual_parameter_easier
batch_load(t.r01) loads one section; batch_load(t) loads everything


## Loading a Graph

`load()` runs the AST backend to extract facts (routes, input accesses,
call edges) and builds a queryable `AnalysisGraph`. Results are cached.

In [5]:
g = load(target)
g

Methods,Rule,Sources,Keys,Flags
POST,/orders,form,"delivery_address, items",
POST,/cart//items,json,item_id,
POST,/cart//checkout,"form, json",,MULTI-SOURCE
POST,/e2e/balance,json,"balance, user_id",


In [6]:
# Notebook outputs are richer when you return objects directly.
g.stats()

Metric,Value
nodes,68
edges,117
call_sites,142
routes,10
input_accesses,15
before_requests,0
dict_merges,1
unique_keys,7
unique_sources,3


In [7]:
g.by_key()

Key,Count,Sources,Accessors,Functions,Example
X-API-Key,1,headers,get,validate_api_key,request.headers.get('X-API-Key')
X-E2E-API-Key,1,headers,get,decorated,"request.headers.get('X-E2E-API-Key', '')"
delivery_address,1,form,get,create_new_order,request.form.get('delivery_address')
item_id,1,json,get,add_item_to_cart_endpoint,request.json.get('item_id')
user_id,1,json,get,e2e_balance,payload.get('user_id')
balance,1,json,get,e2e_balance,payload.get('balance')
items,2,form,getlist,"check_price_and_availability, get_order_items",data.getlist('items')


The graph object itself now renders as the interactive workbench,
even before any findings exist. Use this for open-ended exploration
of routes, call paths, and backend-extracted facts.

In [8]:
Explorer(g)

## Scanning for Vulnerabilities

`scan()` extracts the graph and runs all detection rules. The result
contains both `findings` and the underlying `graph`.

In [9]:
r = scan(target)
r

ScanResult(e03_order_overwrite: 10 routes, 15 accesses, 2 findings)

In [10]:
show(r.findings[0])

[MEDIUM] CONF-005: Dict merge with user-controlled data: {**user_data, **safe_order_data}
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:200
  Endpoint: POST /cart/<cart_id>/checkout (routes.checkout_cart)
  In routes.checkout_cart, a dict merge unpacks user-controlled data alongside other values. User data is unpacked before safe data (safe values win on collision). Fields like 'total', 'user_id', or 'order_id' in the user data could overwrite computed values.
  Evidence:
    - {**user_data, **safe_order_data} at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:200


`show(...)` is the notebook-oriented formatter. `print(...)` uses the
plain string representation of the underlying Python object.

In [11]:
print(r.findings[0])
show(r.findings[0])
r.findings[0]

[MEDIUM] CONF-005: Dict merge with user-controlled data: {**user_data, **safe_order_data} at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:200
[MEDIUM] CONF-005: Dict merge with user-controlled data: {**user_data, **safe_order_data}
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:200
  Endpoint: POST /cart/<cart_id>/checkout (routes.checkout_cart)
  In routes.checkout_cart, a dict merge unpacks user-controlled data alongside other values. User data is unpacked before safe data (safe values win on collision). Fields like 'total', 'user_id', or 'order_id' in the user data could overwrite computed values.
  Evidence:
    - {**user_data, **safe_order_data} at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:200


Finding(rule_id='CONF-005', title='Dict merge with user-controlled data: {**user_data, **safe_order_data}', description="In routes.checkout_cart, a dict merge unpacks user-controlled data alongside other values. User data is unpacked before safe data (safe values win on collision). Fields like 'total', 'user_id', or 'order_id' in the user data could overwrite computed values.", severity=<Severity.MEDIUM: 'medium'>, location=Location(file='/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py', line=200, col=37), evidence=[DictMergeFact(function_qualname='routes.checkout_cart', location=Location(file='/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py', line=200, col=37), sources=('user_data', 'safe_order_data'), raw_code='{**user_data, **safe_order_data}')], endpoint=RouteFact(handler_name='checkout_cart', handler_qualname='routes.checkout_cart', route_kind=RouteKind.DECORATOR, rule='/cart/<cart_id>/checkout', methods=('POST',), blueprint='bp', location=Location(file='/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py', line=169, col=1), raw_code="bp.route('/cart/<cart_id>/checkout', methods=['POST'])", notes=()), details={'sources': ['user_data', 'safe_order_data'], 'user_controlled_position': 0})

## Interactive Exploration

The `Explorer` widget is the main workbench. It accepts a graph,
a scan result, or a graph plus explicit findings.

In [12]:
Explorer(g)

In [13]:
Explorer(r)

## Querying the Graph

Helper functions let you filter and explore the extracted data
without manually iterating.

In [14]:
# All form-source accesses
accesses(g, source="form")

Function,Source,Accessor,Key,Location,Code
routes.create_new_order,form,get,delivery_address,routes.py:92,request.form.get('delivery_address')
routes.create_new_order,form,direct,,routes.py:92,request.form
routes.checkout_cart,form,direct,,routes.py:181,request.form
utils.check_price_and_availability,form,getlist,items,utils.py:73,data.getlist('items')
utils.get_order_items,form,getlist,items,utils.py:87,data.getlist('items')


In [15]:
# Keys with divergent access patterns (different sources or accessors)
diff_keys(g)

Key,Sources,Accessors,Access Count


In [16]:
# Detailed view of a specific endpoint
endpoint_detail("create_new_order", g)

Handler,Route,Methods,Accesses,Sources,Keys,Merges
routes.create_new_order,/orders,POST,4,form,"delivery_address, items",0


## Multi-Exercise Scanning

Scan across an entire section to see how vulnerabilities evolve
from baseline through exploitation to the fixed version.

In [17]:
# Load all r01 exercises at once
graphs = batch_load(t.r01)
print(f"Loaded {len(graphs)} exercises")

# Run all rules across all exercises
from confusion_sast.detection.rules import run_all_rules
results = BatchResult({name: run_all_rules(g) for name, g in graphs.items()})
results

Loaded 9 exercises


Exercise,Findings,Rules,Severity
r01/e00,0,--,--
r01/e01,1,CONF-002,HIGH
r01/e02,2,"CONF-001, CONF-004","HIGH, MEDIUM"
r01/e03,2,"CONF-005, CONF-007",MEDIUM
r01/e04,3,"CONF-001, CONF-005, CONF-007","HIGH, MEDIUM"
r01/e05,3,"CONF-001, CONF-005, CONF-007","HIGH, MEDIUM"
r01/e06,7,"CONF-001, CONF-005, CONF-007","HIGH, MEDIUM"
r01/e07,3,"CONF-001, CONF-005, CONF-007","HIGH, MEDIUM"
r01/e08,3,"CONF-001, CONF-005, CONF-007","HIGH, MEDIUM"


In [18]:
# Browse batch results exercise-by-exercise with the same workbench UX
BatchExplorer(graphs, results)

## Individual Widgets

Use these when you want one part of the workbench in isolation.

In [19]:
# Code viewer: show the source location of a finding
viewer = CodeViewer()
viewer.show_finding(r.findings[0])

In [20]:
# Code path view: inspect the route-to-evidence context
cp = CodePathView(target_path=target.path)
cp.show_finding(g, r.findings[0])
cp.widget

In [21]:
# Graph view: focus the graph on a specific finding
gv = GraphView(g, layout="dagre")
gv.focus_finding(r.findings[0])
gv.widget

In [22]:
# Endpoint browser: open-ended graph exploration
et = EndpointTable(g)
et.widget